In [1]:
import numpy as np
import psutil
import subprocess
import random as rd
import pickle
import gc
from qiskit.quantum_info import SparsePauliOp
from qiskit_nature.second_q.mappers import LogarithmicMapper

def memory_usage(message: str = 'debug'):
    # this memory_usage function is imported from https://jybaek.tistory.com/895
    # current process RAM usage
    p = psutil.Process()
    rss = p.memory_info().rss / 2 ** 30 # Bytes to GiB
    print(f"[{message}] memory usage: {rss: 10.5f} GiB")


def single_line (lines):
    return ''.join(lines.splitlines())

In [2]:
n_x = 6
n_y = 1
n_site = n_x * n_y
n_qubit = n_site
dim = 2**n_qubit

n_qubit_circuit = n_qubit + 1

n_dimer = n_site//2

core  = 0
cores = 1

In [3]:
seed_transpiler = 42

In [4]:
from qiskit import QuantumRegister, QuantumCircuit
qr_circuit = QuantumRegister(n_qubit_circuit,name='q')
#qr = qr_circuit[1:]
#qm = qr_circuit[0]
#indx_qm = 0
indx_qm = n_qubit//2
qm = qr_circuit[indx_qm]
qr = qr_circuit[:indx_qm] + qr_circuit[indx_qm+1:]
#print(qm,qr)

In [5]:
J                   = 1.0
Delta               = -1.0
h                   = 0.0

In [6]:
n_inter             = 2
t_inter_max         = 1.0
t_inters            = [0.0, 1.0]
hamiltonians        = []
mapper = LogarithmicMapper()
for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    hamiltonian_list = []
    # intra-dimer terms
    for i in range(0,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*Delta/4)
        hamiltonian_list.append(term)
        term = ('Z',[ii],-h/2)
        hamiltonian_list.append(term)
        term = ('Z',[jj],-h/2)
        hamiltonian_list.append(term)
    # inter-dimer terms
    for i in range(1,n_qubit,2):
        ii = i
        #jj = (i+1)%n_qubit # periodic boundary condition
        jj = (i+1) # open boundary condition
        if (jj>=n_qubit):
            continue
        term = ('XX',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('YY',[ii,jj],-J*t_inter/4)
        hamiltonian_list.append(term)
        term = ('ZZ',[ii,jj],-J*t_inter*Delta/4)
        hamiltonian_list.append(term)
    print(hamiltonian_list)
    hamiltonian = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonians.append(hamiltonian.simplify())

    if (core==0):
        print(t_inter, single_line(str(hamiltonians[i_inter])))
        print('')

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [4, 5], -0.25), ('YY', [4, 5], -0.25), ('ZZ', [4, 5], 0.25), ('Z', [4], -0.0), ('Z', [5], -0.0), ('XX', [1, 2], -0.0), ('YY', [1, 2], -0.0), ('ZZ', [1, 2], 0.0), ('XX', [3, 4], -0.0), ('YY', [3, 4], -0.0), ('ZZ', [3, 4], 0.0)]
0.0 SparsePauliOp(['IIIIXX', 'IIIIYY', 'IIIIZZ', 'IIXXII', 'IIYYII', 'IIZZII', 'XXIIII', 'YYIIII', 'ZZIIII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])

[('XX', [0, 1], -0.25), ('YY', [0, 1], -0.25), ('ZZ', [0, 1], 0.25), ('Z', [0], -0.0), ('Z', [1], -0.0), ('XX', [2, 3], -0.25), ('YY', [2, 3], -0.25), ('ZZ', [2, 3], 0.25), ('Z', [2], -0.0), ('Z', [3], -0.0), ('XX', [4, 5], -0.25), ('YY', [4, 5], -0.25), ('ZZ', [4, 5], 0.25), ('Z', [4], -0.0), ('Z', [5], -0.0), ('XX', [1, 2

In [7]:
init_circuit = QuantumCircuit(qr_circuit)
# init circuit for staring from dimers
n_dimer_half = n_dimer//2
index_attached_to_qm = n_qubit//2-1

#  geometry ; qm -- qr[n_qubit//2-1] -- ... -- qr[1] -- qr[0]
#                   \--qr[n_qubit//2] -- ... -- qr[n_qubit-1]
# apply cxs first
init_circuit.cx(qm,qr[index_attached_to_qm])
for i_qubit in range(index_attached_to_qm,1,-1):
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
for i_qubit in range(index_attached_to_qm,n_qubit-2):
    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])


# apply chs + z + cx
for i_dimer in range(n_dimer_half,1,-1):
    i_qubit = 2*i_dimer-1
    init_circuit.ch(qr[i_qubit],qr[i_qubit+1])
    init_circuit.z(qr[i_qubit+1])
    init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])

for i_dimer in range(n_dimer_half+1,n_dimer):
    i_qubit = 2*i_dimer-1
    init_circuit.ch(qr[i_qubit+1],qr[i_qubit])
    init_circuit.z(qr[i_qubit])
    init_circuit.cx(qr[i_qubit],qr[i_qubit-1])
# boundaries
init_circuit.ch(qr[1],qr[0])
init_circuit.cx(qr[0],qr[1])

init_circuit.ch(qr[-2],qr[-1])
init_circuit.cx(qr[-1],qr[-2])

#
#for i_dimer in range(1,n_dimer_half+1):
#    i_qubit = 2*(i_dimer+n_dimer_half)
#    init_circuit.s(qr[i_qubit+1])
#    init_circuit.h(qr[i_qubit+1])
#    init_circuit.t(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit],qr[i_qubit+1])
#    init_circuit.tdg(qr[i_qubit+1])
#    if (i_dimer<n_dimer_half):
#        init_circuit.sx(qr[i_qubit+1])
#        init_circuit.cx(qr[i_qubit+1],qr[i_qubit+2])
#        init_circuit.x(qr[i_qubit+1])
#        init_circuit.h(qr[i_qubit+1])
#    else:
#        init_circuit.h(qr[i_qubit+1])
#        init_circuit.sdg(qr[i_qubit+1])
#    init_circuit.cx(qr[i_qubit+1],qr[i_qubit])




print(init_circuit)
init_circuit_inv = init_circuit.inverse()
print(init_circuit_inv)

               ┌───┐                    
q_0: ──────────┤ H ├──■─────────────────
          ┌───┐└─┬─┘┌─┴─┐               
q_1: ─────┤ X ├──■──┤ X ├───────────────
     ┌───┐└─┬─┘     └───┘          ┌───┐
q_2: ┤ X ├──■────■─────────────────┤ X ├
     └─┬─┘       │                 └─┬─┘
q_3: ──■─────────┼───────────────────┼──
               ┌─┴─┐     ┌───┐┌───┐  │  
q_4: ──────────┤ X ├──■──┤ H ├┤ Z ├──■──
               └───┘┌─┴─┐└─┬─┘└───┘┌───┐
q_5: ───────────────┤ X ├──■────■──┤ X ├
                    └───┘     ┌─┴─┐└─┬─┘
q_6: ─────────────────────────┤ H ├──■──
                              └───┘     
          ┌───┐                         
q_0: ──■──┤ H ├─────────────────────────
     ┌─┴─┐└─┬─┘               ┌───┐     
q_1: ┤ X ├──■─────────────────┤ X ├─────
     ├───┤                    └─┬─┘┌───┐
q_2: ┤ X ├─────────────────■────■──┤ X ├
     └─┬─┘                 │       └─┬─┘
q_3: ──┼───────────────────┼─────────■──
       │  ┌───┐┌───┐     ┌─┴─┐          
q_4: ──■──┤ Z ├┤

In [8]:
n_hamiltonians = len(hamiltonians)
if (core==0):
    print('# Hamiltonian differences')
hamiltonian_diffs = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs.append((hamiltonians[alpha+1]-hamiltonians[alpha]).simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs[alpha])))

if (core==0):
    print('# Hamiltonian differences_list')
hamiltonian_diffs_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_list.append(hamiltonian_diffs[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_list[alpha])

if (core==0):
    print('# Reduced Hamiltonian differences_list')
hamiltonian_diffs_reduced = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_list = []
    factor_XX = 2 * 2 # 1 for XX, 1 for YY, 2 for symmetric two bonds
    # for 6 site
    ii = index_attached_to_qm+1
    jj = index_attached_to_qm+2
    term = ('XX',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])/4*factor_XX)
    hamiltonian_list.append(term)
    factor_ZZ = 2 # 2 for symmetric two bonds
    term = ('ZZ',[ii,jj],-J*(t_inters[alpha+1]-t_inters[alpha])*Delta/4*factor_ZZ)
    hamiltonian_list.append(term)
    print(hamiltonian_list)
    dH = SparsePauliOp.from_sparse_list(hamiltonian_list,num_qubits=n_qubit)
    hamiltonian_diffs_reduced.append(dH.simplify())
    if (core==0):
        print(alpha, single_line(str(hamiltonian_diffs_reduced[alpha])))
if (core==0):
    print('# Hamiltonian differences_reduced_list')
hamiltonian_diffs_reduced_list = []
for alpha in range(n_hamiltonians-1):
    hamiltonian_diffs_reduced_list.append(hamiltonian_diffs_reduced[alpha].to_list())
    if (core==0):
        print(hamiltonian_diffs_reduced_list[alpha])

# Hamiltonian differences
0 SparsePauliOp(['IIIXXI', 'IIIYYI', 'IIIZZI', 'IXXIII', 'IYYIII', 'IZZIII'],              coeffs=[-0.25+0.j, -0.25+0.j,  0.25+0.j, -0.25+0.j, -0.25+0.j,  0.25+0.j])
# Hamiltonian differences_list
[('IIIXXI', (-0.25+0j)), ('IIIYYI', (-0.25+0j)), ('IIIZZI', (0.25+0j)), ('IXXIII', (-0.25+0j)), ('IYYIII', (-0.25+0j)), ('IZZIII', (0.25+0j))]
# Reduced Hamiltonian differences_list
[('XX', [3, 4], -1.0), ('ZZ', [3, 4], 0.5)]
0 SparsePauliOp(['IXXIII', 'IZZIII'],              coeffs=[-1. +0.j,  0.5+0.j])
# Hamiltonian differences_reduced_list
[('IXXIII', (-1+0j)), ('IZZIII', (0.5+0j))]


In [9]:
nmc = 300
beta = 2.0
n_shot = 1024

n_obs = 3
# 0; norm, 1; dE1, 2; dE2
O_timelists         = [[[[] for _ in range(nmc)] for _ in range(n_obs)] for _ in range(n_hamiltonians)]

exact_dir = '../exact'
with open(exact_dir+'/XXZ6.time.binary','rb') as file_:
    [O_timelists] = pickle.load(file_)

In [10]:
observable = SparsePauliOp.from_sparse_list([("Z", [indx_qm], 1)], num_qubits=n_qubit_circuit)
print(observable)

SparsePauliOp(['IIIZIII'],
              coeffs=[1.+0.j])


In [11]:
def Apply_R_XXplusYYplusZZ(theta_x,theta_y,theta_z, jj, qc_: QuantumCircuit):
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(theta_z,qr[jj])
    qc_.rz(theta_x+np.pi/2,qr[jj+1])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.h(qr[jj+1])
    qc_.rz(-theta_y,qr[jj])
    qc_.cx(qr[jj+1],qr[jj])
    qc_.sx(qr[jj])
    qc_.sxdg(qr[jj+1])
    


In [12]:
qc = QuantumCircuit(qr_circuit)
Apply_R_XXplusYYplusZZ(1.0, 2.0, 3.0, 0, qc)
Apply_R_XXplusYYplusZZ(1.0, 2.0, 3.0, 2, qc)
Apply_R_XXplusYYplusZZ(1.0, 2.0, 3.0, 1, qc)
print(qc)

     ┌───┐┌───────┐              ┌───┐┌────────┐┌───┐ ┌────┐               »
q_0: ┤ X ├┤ Rz(3) ├──────────────┤ X ├┤ Rz(-2) ├┤ X ├─┤ √X ├───────────────»
     └─┬─┘└─┬───┬─┘┌────────────┐└─┬─┘└─┬───┬──┘└─┬─┘┌┴────┴┐┌───┐┌───────┐»
q_1: ──■────┤ H ├──┤ Rz(2.5708) ├──■────┤ H ├─────■──┤ √Xdg ├┤ X ├┤ Rz(3) ├»
     ┌───┐┌─┴───┴─┐└────────────┘┌───┐┌─┴───┴──┐┌───┐└┬────┬┘└─┬─┘└─┬───┬─┘»
q_2: ┤ X ├┤ Rz(3) ├──────────────┤ X ├┤ Rz(-2) ├┤ X ├─┤ √X ├───■────┤ H ├──»
     └─┬─┘└───────┘              └─┬─┘└────────┘└─┬─┘ └────┘        └───┘  »
q_3: ──┼───────────────────────────┼──────────────┼────────────────────────»
       │    ┌───┐  ┌────────────┐  │    ┌───┐     │  ┌──────┐              »
q_4: ──■────┤ H ├──┤ Rz(2.5708) ├──■────┤ H ├─────■──┤ √Xdg ├──────────────»
            └───┘  └────────────┘       └───┘        └──────┘              »
q_5: ──────────────────────────────────────────────────────────────────────»
                                                                           »

In [13]:
def ApplyConsecutiveTrotterEvolution(times, alphas, n_trotters, indx, qc_:QuantumCircuit):
    n_time = len(times)
    if (n_time<1):
        return
    thetas_xx_inter = [0.0 for _ in range(n_time)]
    thetas_yy_inter = [0.0 for _ in range(n_time)]
    thetas_zz_inter = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_yy_inter[i_time] = -J/2.0 * dt * t_inters[alphas[i_time]]
        thetas_zz_inter[i_time] = -J/2.0 * dt * Delta * t_inters[alphas[i_time]]

    thetas_xx_intra = [0.0 for _ in range(n_time)]
    thetas_yy_intra = [0.0 for _ in range(n_time)]
    thetas_zz_intra = [0.0 for _ in range(n_time)]

    for i_time in range(n_time):
        dt = times[i_time]/n_trotters[i_time]
        thetas_xx_intra[i_time] = -J/2.0 * dt
        thetas_yy_intra[i_time] = -J/2.0 * dt
        thetas_zz_intra[i_time] = -J/2.0 * dt * Delta
    if (indx==0):
        # inter-dimer evolution first
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[0]/2.0,
                                   thetas_yy_inter[0]/2.0,
                                   thetas_zz_inter[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                       thetas_yy_intra[i_time],
                                       thetas_zz_intra[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_inter[i_time]+thetas_xx_inter[i_time+1])/2.0,
                                       (thetas_yy_inter[i_time]+thetas_yy_inter[i_time+1])/2.0,
                                       (thetas_zz_inter[i_time]+thetas_zz_inter[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                   thetas_yy_intra[-1],
                                   thetas_zz_intra[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_inter[-1])/2.0,
                                   (thetas_yy_inter[-1])/2.0,
                                   (thetas_zz_inter[-1])/2.0,
                                   i_qubit, qc_)
    elif (indx==1):
        # intra-dimer evolution first
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_intra[0]/2.0,
                                   thetas_yy_intra[0]/2.0,
                                   thetas_zz_intra[0]/2.0,
                                   i_qubit, qc_)
        for i_time in range(n_time-1):
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                       thetas_yy_inter[i_time],
                                       thetas_zz_inter[i_time],
                                       i_qubit, qc_)
            for i_trotter in range(n_trotters[i_time]-1):
                # intra-dimer evolution
                for i_qubit in range(0,n_qubit,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_intra[i_time],
                                           thetas_yy_intra[i_time],
                                           thetas_zz_intra[i_time],
                                           i_qubit, qc_)
                # inter-dimer evolution
                for i_qubit in range(1,n_qubit-1,2):
                    Apply_R_XXplusYYplusZZ(thetas_xx_inter[i_time],
                                           thetas_yy_inter[i_time],
                                           thetas_zz_inter[i_time],
                                           i_qubit, qc_)
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ((thetas_xx_intra[i_time]+thetas_xx_intra[i_time+1])/2.0,
                                       (thetas_yy_intra[i_time]+thetas_yy_intra[i_time+1])/2.0,
                                       (thetas_zz_intra[i_time]+thetas_zz_intra[i_time+1])/2.0,
                                       i_qubit, qc_)
        # last evolution
        # inter-dimer evolution
        for i_qubit in range(1,n_qubit-1,2):
            Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                   thetas_yy_inter[-1],
                                   thetas_zz_inter[-1],
                                   i_qubit, qc_)
        for i_trotter in range(n_trotters[-1]-1):
            # intra-dimer evolution
            for i_qubit in range(0,n_qubit,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_intra[-1],
                                       thetas_yy_intra[-1],
                                       thetas_zz_intra[-1],
                                       i_qubit, qc_)
            # inter-dimer evolution
            for i_qubit in range(1,n_qubit-1,2):
                Apply_R_XXplusYYplusZZ(thetas_xx_inter[-1],
                                       thetas_yy_inter[-1],
                                       thetas_zz_inter[-1],
                                       i_qubit, qc_)
        # intra-dimer evolution
        for i_qubit in range(0,n_qubit,2):
            Apply_R_XXplusYYplusZZ((thetas_xx_intra[-1])/2.0,
                                   (thetas_yy_intra[-1])/2.0,
                                   (thetas_zz_intra[-1])/2.0,
                                   i_qubit, qc_)

In [14]:
# read exact values
nsec = n_qubit//2 + 1
exact_dir = '../exact'
norms_exact = np.zeros((nsec,n_hamiltonians), dtype=float)
eigen_energies_exact = np.zeros((nsec,n_hamiltonians),dtype=float)
with open(exact_dir + '/exact.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        for isec in range(nsec):
            norms_exact[isec,alpha] = float(ls[isec+1])
            eigen_energies_exact[isec,alpha] = float(ls[isec+nsec+1])
        alpha += 1
for isec in range(nsec):
    for alpha in range(n_hamiltonians):
        print(isec,alpha,norms_exact[isec,alpha],eigen_energies_exact[isec,alpha])

0 0 1.0 0.75
0 1 1.0 1.25
1 0 1.0 -0.25
1 1 0.15550211698203645 -0.6160254037844385
2 0 1.0 -1.25
2 1 0.19329015903038524 -2.0019953568985325
3 0 1.0 -2.2500000000000004
3 1 0.8502049469634537 -2.4935771338879253


In [15]:
from qiskit.quantum_info import Operator
qc_ref = QuantumCircuit(qr_circuit)
qc_ref.rxx(1.0,qr[0],qr[1])
qc_ref.ryy(2.0,qr[0],qr[1])
qc_ref.rzz(3.0,qr[0],qr[1])

u_ref = Operator(qc_ref).data
#print(qc_ref.decompose())
#print(u_ref)

qc = QuantumCircuit(qr_circuit)
Apply_R_XXplusYYplusZZ(1.0,2.0,3.0,0,qc)
print(qc)

u = Operator(qc).data
phase = np.pi/4.0 # = np.angle(u_ref[0,0]/u[0,0])
#print('')
print(np.max(np.abs(u*np.exp(1j*phase)-u_ref)))

     ┌───┐┌───────┐              ┌───┐┌────────┐┌───┐ ┌────┐ 
q_0: ┤ X ├┤ Rz(3) ├──────────────┤ X ├┤ Rz(-2) ├┤ X ├─┤ √X ├─
     └─┬─┘└─┬───┬─┘┌────────────┐└─┬─┘└─┬───┬──┘└─┬─┘┌┴────┴┐
q_1: ──■────┤ H ├──┤ Rz(2.5708) ├──■────┤ H ├─────■──┤ √Xdg ├
            └───┘  └────────────┘       └───┘        └──────┘
q_2: ────────────────────────────────────────────────────────
                                                             
q_3: ────────────────────────────────────────────────────────
                                                             
q_4: ────────────────────────────────────────────────────────
                                                             
q_5: ────────────────────────────────────────────────────────
                                                             
q_6: ────────────────────────────────────────────────────────
                                                             
2.0407899217551515e-16


In [16]:
def NumberOfTrotterSteps(alpha):
    return 2
if (core==0):
    print('# of Trotter Steps for each alpha')
    for alpha_ in range(1,n_hamiltonians):
        print(NumberOfTrotterSteps(alpha_))

# of Trotter Steps for each alpha
2


In [17]:
#from qiskit_ibm_runtime import QiskitRuntimeService
# 
## Load saved credentials
#service = QiskitRuntimeService()
#backend_name = "ibm_torino" 
#backend = service.backend(name=backend_name)

# use fake backend
from qiskit_ibm_runtime.fake_provider import FakeTorino
backend = FakeTorino()
from qiskit_ibm_runtime import EstimatorOptions
from qiskit_ibm_runtime import EstimatorV2 as Estimator
options = EstimatorOptions()
options.default_shots = n_shot
options.resilience_level = 1
options.dynamical_decoupling.enable = True
options.dynamical_decoupling.sequence_type = "XY4"
estimator = Estimator(backend, options=options)
max_circuits = 300 # backend.max_circuits

In [18]:
initial_layout = [95,96,97,110,98,99,92] # seems to best at 241111, 08:12 PM

In [19]:
## Single trotter test
#from qiskit.quantum_info import Operator
##times = [2.0,3.0]
##n_trotters = [2,2]
##alphas = [1,2]
#times = [2.0]
#n_trotters = [2]
#alphas = [1]
#indx = 1
#qc = QuantumCircuit(qr_circuit)
#qc.h(qm)
#for inst in init_circuit.data:
#    qc.append(inst.operation, inst.qubits, inst.clbits)
#ApplyConsecutiveTrotterEvolution(times,alphas,n_trotters,indx,qc)
#for inst in init_circuit_inv.data:
#    qc.append(inst.operation, inst.qubits, inst.clbits)
#qc.h(qm)
#
#unitary = Operator(qc).data
#print(np.abs(unitary[0,0]))
#Z_mat = observable.to_matrix()
#evol_real = np.real((unitary.conj().T@Z_mat@unitary)[0,0])
#print(evol_real)
##print(qc)
#
#qc = QuantumCircuit(qr_circuit)
#qc.h(qm)
#for inst in init_circuit.data:
#    qc.append(inst.operation, inst.qubits, inst.clbits)
#ApplyConsecutiveTrotterEvolution(times,alphas,n_trotters,indx,qc)
#for inst in init_circuit_inv.data:
#    qc.append(inst.operation, inst.qubits, inst.clbits)
#qc.sdg(qm)
#qc.h(qm)
#
#unitary = Operator(qc).data
#Z_mat = observable.to_matrix()
#evol_imag = np.real((unitary.conj().T@Z_mat@unitary)[0,0])
#print(evol_imag)
#
#evol = evol_real + 1j * evol_imag
#print(evol)
#
## exact value
#eps = 0.0
#isec = nsec-1
#phi = eigen_vectors_exact[isec][0,:,0]
##print(phi)
#u_exact = ExactEvolution(isec, alphas[0], eps, times[0])
##u_exact = ExactEvolution(isec, alphas[1], eps, times[1])@u_exact
#u_trotter = TrotterEvolution(isec,alphas[0],times[0],eps,n_trotters[0],indx)
##u_trotter = TrotterEvolution(isec,alphas[1],times[1],eps,n_trotters[1],indx)@u_trotter
##u_trotter = ExactEvolution_inter(isec,alphas[0],eps,times[0]) # weird.. how?
#evol_exact = phi.conj()@u_exact@phi
#evol_trotter = phi.conj()@u_trotter@phi
#print(evol_exact,evol_trotter)
#print(np.abs(evol_trotter/evol))
#

In [20]:
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager
pm3 = generate_preset_pass_manager(initial_layout=initial_layout, backend=backend, optimization_level=3, seed_transpiler=seed_transpiler)
pm1 = generate_preset_pass_manager(backend=backend, optimization_level=1, seed_transpiler=seed_transpiler)

In [21]:
# make a circuit
def ApplyControlledPauliGate_bare(paulis, qc_, qr_, qm_):
    for i in range(len(qr_)):
        p = paulis[i]
        if p=="I":
            continue
        if p=="X":
            qc_.cx(qm_,qr_[i])
        if p=="Y":
            qc_.cy(qm_,qr_[i])
        if p=="Z":
            qc_.cz(qm_,qr_[i])
nhd = len(hamiltonian_diffs_reduced_list[0]) # same difference elements for all
circuits = []
# CXX
circuit = QuantumCircuit(qr_circuit)
circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
circuit.cx(qm,qr[index_attached_to_qm])
circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
circuits.append(circuit)
# CZZ
circuit = QuantumCircuit(qr_circuit)
circuit.h(qr[index_attached_to_qm])
circuit.h(qr[index_attached_to_qm-1])
circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
circuit.cx(qm,qr[index_attached_to_qm])
circuit.cx(qr[index_attached_to_qm],qr[index_attached_to_qm-1])
circuit.h(qr[index_attached_to_qm-1])
circuit.h(qr[index_attached_to_qm])
circuits.append(circuit)
#for ihd in range(nhd):
#    circuit = QuantumCircuit(qr_circuit)
#    pauli_ = hamiltonian_diffs_reduced_list[alpha-1][ihd][0]
#    for i in range(len(qr)):
#        p = pauli_[-i-1]
#    ApplyControlledPauliGate_bare(pauli_, circuit, qr, qm)
#    circuits.append(circuit)

In [22]:
for circuit in circuits:
    print(circuit)

                    
q_0: ───────────────
     ┌───┐     ┌───┐
q_1: ┤ X ├─────┤ X ├
     └─┬─┘┌───┐└─┬─┘
q_2: ──■──┤ X ├──■──
          └─┬─┘     
q_3: ───────■───────
                    
q_4: ───────────────
                    
q_5: ───────────────
                    
q_6: ───────────────
                    
                              
q_0: ─────────────────────────
     ┌───┐┌───┐     ┌───┐┌───┐
q_1: ┤ H ├┤ X ├─────┤ X ├┤ H ├
     ├───┤└─┬─┘┌───┐└─┬─┘├───┤
q_2: ┤ H ├──■──┤ X ├──■──┤ H ├
     └───┘     └─┬─┘     └───┘
q_3: ────────────■────────────
                              
q_4: ─────────────────────────
                              
q_5: ─────────────────────────
                              
q_6: ─────────────────────────
                              


In [23]:
paulis_opt = []
for circuit in circuits:
    circuit_opt = pm3.run(circuit)
    paulis_opt.append(circuit_opt) 

In [24]:
paulis_opt[0].draw(idle_wires=False)

┌─────────┐┌────┐┌───────┐                                      »
 q_1 -> 96 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────────────────────────────»
           └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐   ┌────┐»
 q_2 -> 97 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─┤ √X ├»
                                        └─────────┘└────┘└───────┘ │ └────┘»
q_3 -> 110 ────────────────────────────────────────────────────────■───────»
                                                                           »
«                         ┌────┐┌─────────┐
« q_1 -> 96 ────────────■─┤ √X ├┤ Rz(π/2) ├
«           ┌─────────┐ │ └────┘└─────────┘
« q_2 -> 97 ┤ Rz(π/2) ├─■──────────────────
«           └─────────┘                    
«q_3 -> 110 ───────────────────────────────
«

In [25]:
paulis_opt[1].draw(idle_wires=False)

»
 q_1 -> 96 ───────────────────────────■───────────────────────────────────■─»
           ┌─────────┐┌────┐┌───────┐ │ ┌────┐┌───────┐   ┌────┐┌───────┐ │ »
 q_2 -> 97 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─┤ √X ├┤ Rz(π) ├─■─┤ √X ├┤ Rz(π) ├─■─»
           └─────────┘└────┘└───────┘   └────┘└───────┘ │ └────┘└───────┘   »
q_3 -> 110 ─────────────────────────────────────────────■───────────────────»
                                                                            »
«                            
« q_1 -> 96 ─────────────────
«           ┌────┐┌─────────┐
« q_2 -> 97 ┤ √X ├┤ Rz(π/2) ├
«           └────┘└─────────┘
«q_3 -> 110 ─────────────────
«

In [26]:
# find the position of CZ[qm,qr]
# for pauli0, it is 7
# for pauli1, it is 6
indx_cz = [0 for _ in range(nhd)]
indx_cz[0] = 7
indx_cz[1] = 6
# check
k = 0
for inst in paulis_opt[0].data:
    if (k==indx_cz[0]):
        print(inst)
    k += 1 
k = 0
for inst in paulis_opt[1].data:
    if (k==indx_cz[1]):
        print(inst)
    k += 1 

CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 110), Qubit(QuantumRegister(133, 'q'), 97)), clbits=())
CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 110), Qubit(QuantumRegister(133, 'q'), 97)), clbits=())


In [27]:
# get a duration
from qiskit.transpiler import InstructionDurations
dur = InstructionDurations.from_backend(backend)
#print(dur)
t_cx_m1 = dur.get('cz',(initial_layout[indx_qm],initial_layout[index_attached_to_qm]))
print(t_cx_m1*dur.dt)

1.04e-07


In [28]:
observable_device = observable.apply_layout(paulis_opt[0].layout)
n_qubit_device = backend.num_qubits
pauli = observable_device.paulis[0]
for k in range(n_qubit_device):
    p = pauli[k]
    if (str(p)=='Z'):
        print(k,p)

110 Z


In [29]:
def ApplyControlledPauliGate(ihd, qc_):
    for inst in paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)
empty_paulis_opt = []
for ihd in range(nhd):
    circuit = QuantumCircuit(qr_circuit)
    circuit = pm3.run(circuit)
    ii = 0
    for inst in paulis_opt[ihd].data:
        #print(inst)
        if ii==indx_cz[ihd]:
            print(inst)
            circuit.barrier()
            circuit.delay(t_cx_m1,initial_layout[2])
            circuit.delay(t_cx_m1,initial_layout[1])
            circuit.barrier()
        else:
            circuit.append(inst.operation, inst.qubits, inst.clbits)
        ii += 1
    empty_paulis_opt.append(circuit)
def ApplyEmptyPauliGate(ihd, qc_):
    for inst in empty_paulis_opt[ihd].data:
        qc_.append(inst.operation, inst.qubits, inst.clbits)

CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 110), Qubit(QuantumRegister(133, 'q'), 97)), clbits=())
CircuitInstruction(operation=Instruction(name='cz', num_qubits=2, num_clbits=0, params=[]), qubits=(Qubit(QuantumRegister(133, 'q'), 110), Qubit(QuantumRegister(133, 'q'), 97)), clbits=())


In [30]:
empty_paulis_opt[0].draw(idle_wires=False)

┌─────────┐┌────┐┌───────┐                              ░ »
q_1 -> 96 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■────────────────────────────░─»
          └─────────┘└────┘└───────┘ │ ┌─────────┐┌────┐┌───────┐ ░ »
q_2 -> 97 ───────────────────────────■─┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─░─»
                                       └─────────┘└────┘└───────┘ ░ »
«          ┌───────────────┐ ░                     ┌────┐┌─────────┐
«q_1 -> 96 ┤ Delay(26[dt]) ├─░───────────────────■─┤ √X ├┤ Rz(π/2) ├
«          ├───────────────┤ ░ ┌────┐┌─────────┐ │ └────┘└─────────┘
«q_2 -> 97 ┤ Delay(26[dt]) ├─░─┤ √X ├┤ Rz(π/2) ├─■──────────────────
«          └───────────────┘ ░ └────┘└─────────┘

In [31]:
empty_paulis_opt[1].draw(idle_wires=False)

░ ┌───────────────┐ ░ »
q_1 -> 96 ───────────────────────────■─────────────────░─┤ Delay(26[dt]) ├─░─»
          ┌─────────┐┌────┐┌───────┐ │ ┌────┐┌───────┐ ░ ├───────────────┤ ░ »
q_2 -> 97 ┤ Rz(π/2) ├┤ √X ├┤ Rz(π) ├─■─┤ √X ├┤ Rz(π) ├─░─┤ Delay(26[dt]) ├─░─»
          └─────────┘└────┘└───────┘   └────┘└───────┘ ░ └───────────────┘ ░ »
«                                             
«q_1 -> 96 ────────────────■──────────────────
«          ┌────┐┌───────┐ │ ┌────┐┌─────────┐
«q_2 -> 97 ┤ √X ├┤ Rz(π) ├─■─┤ √X ├┤ Rz(π/2) ├
«          └────┘└───────┘   └────┘└─────────┘

In [32]:
#circuit = QuantumCircuit(qr_circuit)
#ApplyControlledPauliGate(0,circuit)
#print(circuit)
#ApplyControlledPauliGate(1,circuit)
#print(circuit)

In [33]:
eigen_energies_ref = np.zeros((n_hamiltonians),dtype=float)

for i_inter in range(n_inter):
    t_inter = t_inters[i_inter]
    eigen_energies_ref[i_inter] = - J/4 * Delta * (n_dimer)
    # inter-dimer energies
    eigen_energies_ref[i_inter] += - J/4 * Delta * t_inter* (n_dimer-1) # open boundary condition

In [34]:
print(eigen_energies_ref)

[0.75 1.25]


In [35]:
e_dimer = -0.25 * J * (2.0-Delta)
print (e_dimer)

# read 4 qubit QZMC values
four_qubit_dir = '../../4/ibm_torino'
n_alpha_4 = 1
e_qzmc_4_qubit = 0.0
with open(four_qubit_dir + '/qzmc.norm.E.save','r') as file_:
    alpha = 0
    for line in file_:
        if line.startswith('#'):
            continue
        ls = line.split()
        if (alpha==n_alpha_4):
            e_qzmc_4_qubit = float(ls[-1])
        alpha += 1
print(e_qzmc_4_qubit)

-0.75
-1.639789655187768


In [36]:
norms_qzmc              = np.ones((n_hamiltonians),dtype=float)
eigen_energies_qzmc     = np.zeros((n_hamiltonians),dtype=float)


eigen_energies_qzmc[0]  = e_dimer*n_dimer
print(eigen_energies_qzmc[0])

-2.25


In [37]:
pauli_identity = 'I'*n_qubit

In [38]:
def fake_run_estimator (max_circuits_, estimator_, pubs_):
    len_circuits_ = len(pubs_)
    num_jobs_     = len_circuits_//max_circuits_
    remainder_    = len_circuits_%max_circuits_
    if (remainder_>0):
        num_jobs_ += 1
    i_start_ = 0
    list_out = []
    for _ in range(num_jobs_):
        i_end_ = min(i_start_ + max_circuits_,len_circuits_)
        job_ = estimator_.run(pubs_[i_start_:i_end_])
        job_result = job_.result()
        len_result = len(job_result)
        for i in range(len_result):
            list_out.append(job_result[i].data.evs)
        i_start_ += max_circuits_
    return list_out

In [39]:
def run_estimator (max_circuits_, estimator_, pubs_):
    len_circuits_ = len(pubs_)
    num_jobs_     = len_circuits_//max_circuits_
    remainder_    = len_circuits_%max_circuits_
    if (remainder_>0):
        num_jobs_ += 1
    i_start_ = 0
    job_ids_ = []
    for _ in range(num_jobs_):
        i_end_ = min(i_start_ + max_circuits_,len_circuits_)
        job_ = estimator_.run(pubs_[i_start_:i_end_])
        job_ids_.append(job_.job_id())
        i_start_ += max_circuits_
    return job_ids_

In [40]:
from qiskit.providers import JobStatus
import time as time_lib

def moniter_jobs (service_, job_ids_):
    num_jobs_       = len(job_ids_)
    done_           = False
    problem_        = False
    while not done_:
        done_ = True
        i_start_ = 0
        for ind_job_ in range(num_jobs_):
            job_id_ = job_ids_[ind_job_]
            job_ = service_.job(job_id_)
            if job_.status() is JobStatus.DONE:
                continue
            else:
                done_ = False
                # exit if there is a problem
                if job_.status() in [JobStatus.ERROR, JobStatus.CANCELLED]:
                    s_ = ''
                    s_ += '## There was a problem in submitting job_ids[{ind_job}], '.format(ind_job=ind_job_)
                    s_ +='\n'
                    s_ += '  {}'.format(job_id_)
                    s_ +='\n'
                    print(s_)
                    problem_ = True
                if (problem_):
                    break
        if (problem_):
            break
        time_lib.sleep(30)
    return [done_, problem_]

def accumulate_job_results (service_, job_ids_):
    list_out = []
    for job_id_ in job_ids_:
        job_ = service_.job(job_id_)
        while job_.status() is not JobStatus.DONE:
            time_lib.sleep(30) 
        job_result = job_.result()
        len_result = len(job_result)
        for i in range(len_result):
            list_out.append(job_result[i].data.evs)
    return list_out


In [41]:
from datetime import datetime
from qiskit.circuit import ParameterVector

In [42]:
job_ids_save = [None for _ in range(n_hamiltonians)]
num_jobs_save = [None for _ in range(n_hamiltonians)]

In [43]:
# eps = eigen_energies_qzmc[0]
# first-order perturbation correction = 0
# This is better estimate of the energy
# eps = e_2 + e_4
eps = e_dimer + e_qzmc_4_qubit

#comm.Barrier()
for alpha in range(1,n_hamiltonians):
    start_time = datetime.now()
    circuits = []
    times = ParameterVector('t', 2*alpha-1)

    # dE1
    nhd = len(hamiltonian_diffs_reduced_list[alpha-1])
    for ihd in range(nhd):
        # norm for ihd
        # implement quantum circuits by parts
        circuit_segment = [[None] * 2 for _ in range(2)] 
        for indx in range(2):
            circuit = QuantumCircuit(qr_circuit)
            circuit.h(qm)
            for inst in init_circuit.data:
                circuit.append(inst.operation, inst.qubits, inst.clbits)
            i_time  = 0
            phase   = 0.0
            # \mathcal{P}_{\alpha-1}
            evolution_times = []
            alphas = []
            n_trotters = []
            for alpha_ in range(1,alpha):
                alphas.append(alpha_)
                evolution_times.append(times[i_time])
                n_trotters.append(NumberOfTrotterSteps(alpha_))
                phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
                i_time += 1
            ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
            circuit_segment[indx][0] = pm3.run(circuit)

            circuit = QuantumCircuit(qr_circuit)

            evolution_times = []
            alphas = []
            n_trotters = []

            # P_{\alpha}
            alphas.append(alpha)
            evolution_times.append(times[i_time])
            n_trotters.append(NumberOfTrotterSteps(alpha))
            #print(evolution_times,alphas,n_trotters)
            phase += (eps-eigen_energies_ref[alpha]) * times[i_time]
            i_time += 1

            # \mathcal{P}^{\dagger}_{\alpha-1}
            for alpha_ in reversed(range(1,alpha)):
                alphas.append(alpha_)
                evolution_times.append(times[i_time])
                n_trotters.append(NumberOfTrotterSteps(alpha_))
                phase += (eigen_energies_qzmc[alpha_]-eigen_energies_ref[alpha_]) * times[i_time]
                i_time += 1
            ApplyConsecutiveTrotterEvolution(evolution_times,alphas,n_trotters,indx,circuit)
            #print(circuit)

            for inst in init_circuit_inv.data:
                circuit.append(inst.operation, inst.qubits, inst.clbits)

            circuit.rz(phase,qm)
            circuit.h(qm)

            circuit_segment[indx][1] = pm3.run(circuit)
        
        # 1, norm, indx = 0, 1
        for indx in range(2):
            circuit = QuantumCircuit(qr_circuit)
            circuit = pm3.run(circuit)
            for inst in circuit_segment[indx][0].data:
                circuit.append(inst.operation, inst.qubits, inst.clbits)
            ApplyEmptyPauliGate(ihd,circuit)
            for inst in circuit_segment[indx][1].data:
                circuit.append(inst.operation, inst.qubits, inst.clbits)
            circuit_opt = pm1.run(circuit)
            circuits.append(circuit_opt)
            del circuit_opt

        # dE*norm for ihd
        for indx in range(2):
            circuit = QuantumCircuit(qr_circuit)
            circuit = pm3.run(circuit)
            for inst in circuit_segment[indx][0].data:
                circuit.append(inst.operation, inst.qubits, inst.clbits)
            ApplyControlledPauliGate(ihd,circuit)
            for inst in circuit_segment[indx][1].data:
                circuit.append(inst.operation, inst.qubits, inst.clbits)
            circuit_opt = pm1.run(circuit)
            circuits.append(circuit_opt)
            del circuit_opt

    pubs = []

    n_pubs = nhd * nmc * 2 * 2 # 2 for indx, 2 for norm
    n_pubs_for_ = [0 for _ in range(cores)]
    remainder         = n_pubs%cores
    for i_core in range(cores):
        n_pubs_for_[i_core] = n_pubs//cores
        if (i_core<remainder):
            n_pubs_for_[i_core] += 1
    if (core==0 and alpha==1):
        print('# of different quantum circuits to run = ', n_pubs)

    i_start         = sum(n_pubs_for_[:core])
    i_end           = i_start + n_pubs_for_[core]
    ind_pub         = 0
    indx_circuit    = 0
    ## dE1
    for ihd in range(nhd):
        # norm, indx = 0, 1
        i_obs = 0
        for indx in range(2):
            for imc in range(nmc):
                my_turn = ind_pub>=i_start and ind_pub<i_end
                if (my_turn):
                    pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
                ind_pub += 1
            indx_circuit += 1
        # dE*norm, indx = 0, 1
        i_obs = 1
        for indx in range(2):
            for imc in range(nmc):
                my_turn = ind_pub>=i_start and ind_pub<i_end
                if (my_turn):
                    pubs.append((circuits[indx_circuit],observable_device,O_timelists[alpha][i_obs][imc][:-1]))
                ind_pub += 1
            indx_circuit += 1

    result_values_core = [0.0 for _ in range(n_pubs_for_[core])]


    #job_ids = run_estimator(max_circuits, estimator, pubs)
    result = fake_run_estimator(max_circuits, estimator, pubs)
    #job_ids_save[alpha] = job_ids
    #num_jobs_save[alpha] = len(job_ids)
    #with open('XXZ4.job_ids','w') as file_:
    #    for alpha_ in range(1, n_hamiltonians):
    #        s = ''
    #        for job_id in job_ids_save[alpha_]:
    #            s += '  {}'.format(job_id)
    #        s += '\n'
    #        file_.write(s)
    #[done, problem] = moniter_jobs (service, job_ids_save[alpha])
    #result = accumulate_job_results(service, job_ids_save[alpha])

    gc.collect()

    for i in range(n_pubs_for_[core]):
        result_values_core[i] = result[i] # no additional shot errors for real-hardware simulation
    del result

    ### bcast
    #comm.Barrier()
    result_values = []
    for i_core in range(cores):
        if (n_pubs_for_[i_core]==0):
            continue
        result_values_temp = result_values_core
        result_values += result_values_temp
        #comm.Barrier()
    print(result_values)

    norms_1    = np.zeros((nhd,2),dtype=float)
    pauli_norms_1  = np.zeros((nhd,2),dtype=float)
    i_meas = 0
    for ihd in range(nhd):
        # norm, indx = 0, 1
        for indx in range(2):
            for imc in range(nmc):
                norms_1[ihd,indx] += result_values[i_meas]
                i_meas += 1
        # pauli, indx = 0, 1
        for indx in range(2):
            for imc in range(nmc):
                pauli_norms_1[ihd,indx] += result_values[i_meas]
                i_meas += 1
    norms_1 /= nmc
    pauli_norms_1 /= nmc

    print(norms_1)
    print(pauli_norms_1)
    norm_1 = np.sum(norms_1)/(nhd*2)

    # dE1
    dE1 = 0.0
    for ihd in range(nhd):
        norm_avg = 0.0
        for indx in range(2):
            norm_avg += norms_1[ihd,indx]
        pauli_avg = 0.0
        for indx in range(2):
            pauli_avg += pauli_norms_1[ihd,indx]
        pauli_avg /= norm_avg
        coeff = hamiltonian_diffs_reduced_list[alpha-1][ihd][1]
        dE1 += coeff * pauli_avg
    dE1 = dE1.real
    
    eigen_energies_qzmc[alpha] = eigen_energies_qzmc[alpha-1] + dE1
    norms_qzmc[alpha] = norm_1
    del times
    del result_values[:]
    del result_values_core[:]
    gc.collect()

    end_time = datetime.now()
    elapsed = end_time-start_time
    elapsed = elapsed.total_seconds()
    if (core==0):
        st = '# {percent:.1f}%, elapsed time = {elapsed} secs'.format(percent=((alpha)/(n_hamiltonians-1)*100),elapsed=elapsed)
        memory_usage(st)
        print(alpha, norms_qzmc[alpha], eigen_energies_qzmc[alpha]-eigen_energies_exact[-1,alpha])
        if (alpha<n_hamiltonians-1):
            print('precision of the predictor for next', eps-eigen_energies_exact[-1,alpha+1])
        st = '# {percent:.1f}%'.format(percent=((alpha)/(n_hamiltonians-1)*100))
        print(st)



# of different quantum circuits to run =  2400


/home/mchan/Doc/venv_new_qiskit/lib/python3.10/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:187: UserWarning: The resilience_level option has no effect in local testing mode.
  warnings.warn("The resilience_level option has no effect in local testing mode.")
/home/mchan/Doc/venv_new_qiskit/lib/python3.10/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:234: UserWarning: Options {'dynamical_decoupling': {'enable': True, 'sequence_type': 'XY4'}} have no effect in local testing mode.
  warnings.warn(f"Options {options_copy} have no effect in local testing mode.")
/home/mchan/Doc/venv_new_qiskit/lib/python3.10/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:187: UserWarning: The resilience_level option has no effect in local testing mode.
  warnings.warn("The resilience_level option has no effect in local testing mode.")
/home/mchan/Doc/venv_new_qiskit/lib/python3.10/site-packages/qiskit_ibm_runtime/fake_provider/local_service.py:234: UserW

[array(0.63085938), array(0.6640625), array(0.53320312), array(0.65429688), array(0.58984375), array(0.63476562), array(0.60742188), array(0.625), array(0.59960938), array(0.54296875), array(0.578125), array(0.62304688), array(0.69921875), array(0.67578125), array(0.65625), array(0.59570312), array(0.59179688), array(0.58398438), array(0.7265625), array(0.55859375), array(0.70703125), array(0.58789062), array(0.62304688), array(0.64648438), array(0.68554688), array(0.60351562), array(0.63085938), array(0.70507812), array(0.5390625), array(-0.171875), array(0.70703125), array(0.68554688), array(0.64453125), array(0.671875), array(0.64257812), array(0.62109375), array(0.63867188), array(0.61328125), array(0.625), array(0.6796875), array(0.703125), array(0.37890625), array(0.67382812), array(0.65429688), array(0.52148438), array(0.62109375), array(0.62109375), array(0.71679688), array(-0.35742188), array(0.70117188), array(0.62890625), array(0.59765625), array(0.64648438), array(0.6660156

In [99]:
qc_ = qc.assign_parameters([O_timelists[1][0][0][0]])